# Using multiple tools

The loop from the previous lesson does not change. Adding a tool takes four steps:

1. write the function
2. write its schema
3. add the schema to `tools`, the list `run_conversation` sends with every request
4. add a branch for its name in `run_tool`

This lesson adds `set_reminder` (from [`tools.py`](./tools.py)). The fake reminder only prints
and returns `None`, so its `tool_result` content is `"null"`: Claude still gets an answer
and knows the call succeeded.

## Setup

In [1]:
from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv
from tools import (
    add_duration_to_datetime,
    add_duration_to_datetime_schema,
    get_current_datetime,
    get_current_datetime_schema,
    set_reminder,
    set_reminder_schema,
)

load_dotenv()
client = Anthropic()
model = "claude-haiku-4-5"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None, tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")

## The loop

The same as in lesson 06. Only two lines changed: `set_reminder_schema` in `tools` and the
`set_reminder` branch in `run_tool`.

In [2]:
import json

tools = [get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema]


def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:  # noqa: BLE001 - any tool error goes back to Claude
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }
        print(f"  [{tool_request.name}({tool_request.input}) -> {tool_result_block['content']}]")
        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks


def run_conversation(messages):
    while True:
        response = chat(messages, tools=tools)
        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages


def print_history(messages):
    for number, message in enumerate(messages, start=1):
        content = message["content"]
        if isinstance(content, str):
            summary = repr(content)
        else:
            parts = []
            for block in content:
                if isinstance(block, dict):
                    parts.append(block["type"] + (" (error)" if block.get("is_error") else ""))
                else:
                    parts.append(block.type)
            summary = ", ".join(parts)
        print(f"{number}. {message['role']:9} {summary}")

## The course example

The date is a fixed offset, so Claude needs two tools in a row: `add_duration_to_datetime`
for the date math, then `set_reminder` with the result.

In [3]:
messages = []
add_user_message(messages, "Set a reminder for my doctor's appointment. It's 177 days after Jan 1st, 2050.")
run_conversation(messages)
print_history(messages)


  [add_duration_to_datetime({'datetime_str': '2050-01-01', 'duration': 177, 'unit': 'days'}) -> "Monday, June 27, 2050 12:00:00 AM"]


Now I'll set a reminder for your doctor's appointment on June 27, 2050:
----
Setting the following reminder for 2050-06-27T12:00:00:
Doctor's appointment
----
  [set_reminder({'content': "Doctor's appointment", 'timestamp': '2050-06-27T12:00:00'}) -> null]


Perfect! I've set a reminder for your doctor's appointment on **Monday, June 27, 2050** at 12:00 AM. You'll receive a notification at that time.
1. user      "Set a reminder for my doctor's appointment. It's 177 days after Jan 1st, 2050."
2. assistant tool_use
3. user      tool_result
4. assistant text, tool_use
5. user      tool_result
6. assistant text


Look at the timestamp. `add_duration_to_datetime` returned `12:00:00 AM` (midnight), but Claude
passed `2050-06-27T12:00:00` (noon) to `set_reminder`, and then told the user "12:00 AM".
The tools work; the slip is in converting one tool's output into the next tool's input.
If `add_duration_to_datetime` returned ISO, like `set_reminder` expects, there would be no
AM/PM to convert. Reading the history is how you catch this.

## All three tools

A date relative to today needs every tool: `get_current_datetime`, then
`add_duration_to_datetime`, then `set_reminder`. Claude picks the order itself.

In [4]:
messages = []
add_user_message(messages, "Remind me to renew my passport in 2 weeks at 10am.")
run_conversation(messages)
print_history(messages)


  [get_current_datetime({'date_format': '%Y-%m-%d %H:%M:%S'}) -> "2026-10-30 13:40:26"]


Now I'll add 2 weeks to the current date and set the reminder for 10am:
  [add_duration_to_datetime({'datetime_str': '2026-10-30 10:00:00', 'duration': 2, 'unit': 'weeks', 'input_format': '%Y-%m-%d %H:%M:%S'}) -> "Friday, November 13, 2026 10:00:00 AM"]



----
Setting the following reminder for 2026-11-13T10:00:00:
Renew my passport
----
  [set_reminder({'content': 'Renew my passport', 'timestamp': '2026-11-13T10:00:00'}) -> null]


Perfect! I've set a reminder for you to renew your passport on **Friday, November 13, 2026 at 10:00 AM** - exactly 2 weeks from now. You'll receive a notification at that time.
1. user      'Remind me to renew my passport in 2 weeks at 10am.'
2. assistant tool_use
3. user      tool_result
4. assistant text, tool_use
5. user      tool_result
6. assistant tool_use
7. user      tool_result
8. assistant text


Next lesson: fine-grained tool calling (streaming tool arguments).